# Add Indices

## Import libs

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import spyndex
import rioxarray as rxr
import xarray as xr
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

## Define File Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
cropped_dir = os.path.join(base_path, "6_Cropped_Images")
out_dir = os.path.join(base_path, "8_Images_with_Indices")
os.makedirs(out_dir, exist_ok=True)

## Calculate RGB Indices

In [ ]:
input_path = os.path.join(cropped_dir, "250821_resampled_cropped.tif")

In [ ]:
with rasterio.open(input_path) as src:
    meta = src.meta.copy()
    data = src.read() # Shape: (4, Height, Width)
    valid_mask_2d = src.dataset_mask()

nodata_mask = (valid_mask_2d == 0)
valid_mask = ~nodata_mask

r = (data[0] / 255.0).astype(np.float32)
g = (data[1] / 255.0).astype(np.float32)
b = (data[2] / 255.0).astype(np.float32)
r[nodata_mask] = np.nan
g[nodata_mask] = np.nan
b[nodata_mask] = np.nan

In [ ]:
print("R: ",r[~nodata_mask].min(), r[~nodata_mask].mean(), r[~nodata_mask].max())
print("G: ",g[~nodata_mask].min(), g[~nodata_mask].mean(), g[~nodata_mask].max())
print("B: ",b[~nodata_mask].min(), b[~nodata_mask].mean(), b[~nodata_mask].max())

In [ ]:
print("  -> Computing ExG, VARI, and NGRDI...")
idx = spyndex.computeIndex(
    index=["ExG", "VARI", "NGRDI"],
    params={
        "R": r,
        "G": g,
        "B": b
    }
)
exg = idx[0]
vari = idx[1]
ngrdi = idx[2]

In [ ]:
print("ExG:   ", np.nanmin(exg), np.nanmean(exg), np.nanmax(exg))
print("VARI:  ", np.nanmin(vari), np.nanmean(vari), np.nanmax(vari))
print("NGRDI: ", np.nanmin(ngrdi), np.nanmean(ngrdi), np.nanmax(ngrdi))

##  Scale Indices to [0,1]

In [ ]:
# Robist scaling using 1 and 99 percentile

def scale_to_01(array, valid_mask):
    valid_data = array[valid_mask]
    valid_data = valid_data[np.isfinite(valid_data)] 

    scaled = np.full_like(array, fill_value=np.nan, dtype=np.float32)
    
    if len(valid_data) == 0:
        return scaled
        
    vmin, vmax = np.percentile(valid_data, 1), np.percentile(valid_data, 99)
        
    if vmax - vmin < 1e-6:
        scaled[valid_mask] = 0.0
    else:
        scaled[valid_mask] = np.clip((array[valid_mask] - vmin) / (vmax - vmin), 0.0, 1.0)
        
    return scaled

In [ ]:
exg_scaled = scale_to_01(exg, valid_mask)
vari_scaled = scale_to_01(vari, valid_mask)
ngrdi_scaled = scale_to_01(ngrdi, valid_mask)

In [ ]:
print("ExG:   ", np.nanmin(exg_scaled[~nodata_mask]), np.nanmean(exg_scaled[~nodata_mask]), np.nanmax(exg_scaled[~nodata_mask]))
print("VARI:  ", np.nanmin(vari_scaled[~nodata_mask]), np.nanmean(vari_scaled[~nodata_mask]), np.nanmax(vari_scaled[~nodata_mask]))
print("NGRDI: ", np.nanmin(ngrdi_scaled[~nodata_mask]), np.nanmean(ngrdi_scaled[~nodata_mask]), np.nanmax(ngrdi_scaled[~nodata_mask]))

## Plot Indices

In [ ]:
def plot_index_image(index_array, title="Index Bild", cmap="RdYlGn"):
    """
    Plottet ein 2D-Raster-Array (Kanal) als echtes Bild.
    """
    plt.figure(figsize=(10, 10))
    
    gute_werte = index_array[np.isfinite(index_array)]
    vmin, vmax = np.percentile(gute_werte, 2), np.percentile(gute_werte, 98)
    
    im = plt.imshow(index_array, cmap=cmap, vmin=vmin, vmax=vmax)
    
    plt.colorbar(im, fraction=0.046, pad=0.04, label="Index Wert")    
    plt.title(title, fontsize=16)
    plt.axis("off") 
    plt.show()

In [ ]:
# --- ANWENDUNG ---
# Plotte dir deine rohen Kanäle an!
plot_index_image(exg, title="Excess Green (ExG) - Spatial View", cmap="YlGn")
plot_index_image(vari, title="VARI - Spatial View", cmap="RdYlGn")
plot_index_image(ngrdi, title="NGRDI - Spatial View", cmap="RdYlGn")

## Scale CHM

In [ ]:
chm = data[3].astype(np.float32)
chm[nodata_mask] = np.nan
chm_scaled = scale_to_01(chm, valid_mask)

In [ ]:
plot_index_image(chm_scaled, title="CHM - Spatial View", cmap="gray")

## Export and Save

In [ ]:
# Concatenate new array
# Order: 1=R, 2=G, 3=B, 4=CHM, 5=ExG, 6=VARI, 7=NGRDI
new_data = np.stack([
    r, 
    g, 
    b, 
    chm_scaled, 
    exg_scaled, 
    vari_scaled, 
    ngrdi_scaled
], axis=0)

# Replace NaNs with Nodata value
ML_NODATA = -9999.0
new_data[np.isnan(new_data)] = ML_NODATA

# Update Metadata
meta.update({
    "count": 7,              # 7 instead of 4 channels now
    "dtype": "float32",   
    "nodata": ML_NODATA      
})

# SAVE
output_tiff = os.path.join(out_dir, "250821_7_channel.tif")

print("  -> Save new 7-Channel GeoTIFF...")

with rasterio.open(output_tiff, "w", **meta) as dst:
    dst.write(new_data)
    
print(f"✅ Success! Saved to: {os.path.basename(output_tiff)}")
print("Band-Mapping: 1=R, 2=G, 3=B, 4=CHM, 5=ExG, 6=VARI, 7=NGRDI")

## Check scales

In [ ]:
image_path = os.path.join(out_dir, "250821_7_channel.tif")

In [ ]:
with rasterio.open(image_path) as src:
    meta = src.meta.copy()
    new_data = src.read() 

print(new_data.shape)

In [ ]:
# Create an empty list to collect channel statistics
stats_list = []

ML_NODATA = -9999

# Loop through the number of channels
for channel in range(new_data.shape[0]):
    band_data = new_data[channel]
    valid_data = band_data[band_data != ML_NODATA]

    if valid_data.size > 0:
        b_min = np.nanmin(valid_data)
        b_mean = np.nanmean(valid_data)
        b_max = np.nanmax(valid_data)
    else:
        b_min, b_mean, b_max = np.nan, np.nan, np.nan
    
    stats_list.append({
        "Band": f"Band {channel+1}",
        "Min": b_min,
        "Mean": b_mean,
        "Max": b_max,
        "Actual NaNs (-9999.0)": np.sum(band_data==ML_NODATA),
        "Zeros (0.0)": np.sum(band_data == 0.0)
    })

# Convert to DataFrame for beautiful notebook rendering
df_stats = pd.DataFrame(stats_list)
df_stats.set_index("Band", inplace=True)

# Display the beautiful table
df_stats